In [10]:
# === CONFIGURATION ===
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
DISCOUNT = 0.99
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09
CNN_CONV_CHANNELS = [32, 64]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3


PATH_DECENTRALIZED_AG0 = "decen_cnn_-12_reward/model_agent0_step16050000.pt"
PATH_DECENTRALIZED_AG1 = "decen_cnn_-12_reward/model_agent1_step16050000.pt"

In [11]:
import torch
import numpy as np
import sys
sys.path.append("../") 
from models.value_cnn_new import ValueCNNDecentralizedStandard
from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from orchard.environment import MoveAction

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
# Load Models
dec_models = []
for p in [PATH_DECENTRALIZED_AG0, PATH_DECENTRALIZED_AG1]:
    m = ValueCNNDecentralizedStandard(HEIGHT, WIDTH, 0.0, DISCOUNT, CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS, CNN_CONV_CHANNELS, CNN_KERNEL_SIZE).to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    dec_models.append(m)

In [14]:
from tadd_helpers.eval_functions import nearest_policy


def get_vals(state):
    v0 = dec_models[0].get_value(state, agent_pos=state.agent_position(0))
    v1 = dec_models[1].get_value(state, agent_pos=state.agent_position(1))
    return v0, v1, v0+v1

def dissect_avoidance():
    print("Searching for an Agent 0 AVOIDANCE event...")
    state = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    
    # Run until we find an event
    max_steps = 2000
    found_event = False
    
    for _ in range(max_steps):
        if found_event: break
        
        # Only check when Agent 0 acts
        active_idx = 0 
        
        # 1. Is there an opportunity? (Apple 1 step away)
        curr_pos = state.agent_position(active_idx)
        apple_locs = np.argwhere(state.apples > 0)
        
        has_opportunity = False
        target_apple = None
        
        if len(apple_locs) > 0:
            dists = np.abs(apple_locs - curr_pos).sum(axis=1)
            if np.min(dists) == 1:
                has_opportunity = True
                target_apple = apple_locs[np.argmin(dists)]
        
        # 2. What does the Policy choose?
        # We manually scan actions to find the best one
        best_q = -999.0
        best_act_vec = None
        
        move_q_vals = {}
        
        for action in MoveAction:
            new_pos = np.clip(curr_pos + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
            s_mid = state.copy()
            s_mid.set_agent_position(active_idx, new_pos)
            
            # Calculate V_sum(s_mid)
            v0, v1, vsum = get_vals(s_mid)
            # Q = V_sum because reward is implicit in s_mid or 0 for move
            # (Using strict training logic: Policy maximizes Sum(V_mid))
            
            move_q_vals[action] = {
                'vec': action.vector,
                'v0': v0, 'v1': v1, 'sum': vsum, 
                'on_apple': (state.apples[tuple(new_pos)] > 0)
            }
            
            if vsum > best_q:
                best_q = vsum
                best_act_vec = action.vector

        # 3. Did it Avoid?
        if has_opportunity:
            # Did the best action move ONTO the apple?
            chosen_pos = np.clip(curr_pos + best_act_vec, [0,0], [HEIGHT-1, WIDTH-1])
            moved_on_apple = (state.apples[tuple(chosen_pos)] > 0)
            
            if not moved_on_apple:
                print("\n>>> FOUND AVOIDANCE EVENT! Agent 0 refused to pick.")
                print(state)
                print(f"Agent 0 Position: {curr_pos}")
                print(f"Target Apple: {target_apple}")
                
                print("\n--- DECISION ANALYSIS ---")
                print(f"{'Move Type':<15} | {'V_Sum (Policy)':<15} | {'V_0 (Agent 0)':<15} | {'V_1 (Agent 1)':<15}")
                print("-"*70)
                
                # Sort by V_Sum
                sorted_moves = sorted(move_q_vals.items(), key=lambda x: x[1]['sum'], reverse=True)
                
                for act, data in sorted_moves:
                    type_str = f"{act.name}"
                    marker = "<-- CHOSEN" if np.array_equal(act.vector, best_act_vec) else ""
                    
                    print(f"{type_str:<15} | {data['sum']:<15.2f} | {data['v0']:<15.2f} | {data['v1']:<15.2f} {marker}")
                
                # CHECK THEORY
                # Find the best PICK option
                pick_data = next((d for a, d in move_q_vals.items() if d['on_apple']), None)
                chosen_data = next((d for a, d in move_q_vals.items() if np.array_equal(a.vector, best_act_vec)), None)
                
                if pick_data and chosen_data:
                    print("\n--- HYPOTHESIS CHECK ---")
                    print(f"Value of Picking (Ag0): {pick_data['v0']:.2f}")
                    print(f"Value of Waiting (Ag0): {chosen_data['v0']:.2f}")
                    
                    if chosen_data['v0'] > pick_data['v0']:
                        print("✅ CONFIRMED: Agent 0 prefers Waiting over Picking (Greed).")
                    else:
                        print("❌ SURPRISE: Agent 0 wanted to pick, but Agent 1 dragged the score down?")
                        if chosen_data['v1'] > pick_data['v1']:
                             print("   (It was Agent 1 who preferred Ag0 to wait!)")
                
                found_event = True
        
        # Step Env
        # Use random or nearest just to churn state
        churn_idx = state.get_random_agent_id()
        churn_act = nearest_policy(state, churn_idx)
        churn_pos = np.clip(state.agent_position(churn_idx) + churn_act.vector, [0,0], [HEIGHT-1, WIDTH-1])
        state.set_agent_position(churn_idx, churn_pos)
        if state.apples[tuple(churn_pos)] > 0: state.remove_apple_at(churn_pos)
        spawn_apples(state, SPAWN_PROB)
        despawn_apples(state, DESPAWN_PROB)

dissect_avoidance()

Searching for an Agent 0 AVOIDANCE event...

>>> FOUND AVOIDANCE EVENT! Agent 0 refused to pick.
--- Empty State (Grid: 6x6) ---

--- Agent Locations ---
  Agent 0: (0, 0)
  Agent 1: (3, 1)

--- Agents (Count) ---
1 . . . . .
. . . . . .
. . . . . .
. 1 . . . .
. . . . . .
. . . . . .

--- Apples (Count) ---
. 1 . . . .
1 . . . . 1
. . . . . .
. . . . . .
. . 1 . . 1
. . . . . 1
Agent 0 Position: [0 0]
Target Apple: [0 1]

--- DECISION ANALYSIS ---
Move Type       | V_Sum (Policy)  | V_0 (Agent 0)   | V_1 (Agent 1)  
----------------------------------------------------------------------
LEFT            | 18.95           | 19.93           | -0.99           <-- CHOSEN
STAY            | 18.95           | 19.93           | -0.99           
UP              | 18.95           | 19.93           | -0.99           
DOWN            | 18.65           | 19.43           | -0.78           
RIGHT           | 18.47           | 19.48           | -1.01           

--- HYPOTHESIS CHECK ---
Value of Pickin